# FundScope — Analítica de Fondos Mutuos con IA Generativa

**Metodología:** CRISP-ML(Q) · **Fuentes:** Yahoo Finance (precios y titulares) + GDELT (tono de prensa) + SEC N-PORT (bonos) · **IA generativa:** Qwen3 en Ollama (Gemini de respaldo)

Este notebook recorre las fases de CRISP-ML(Q) con **los mismos módulos que la aplicación** (`src/analisis.py` orquesta el flujo).

| Fase | Sección |
|---|---|
| 1. Comprensión del negocio | 1 |
| 2. Comprensión y preparación de datos | 2 y 3 |
| 3. Modelado | 4 |
| 4. Evaluación | 5 |
| 5. Despliegue (prescriptivo) | 6 |
| 6. Monitoreo de calidad | 7 |

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
rutaProyecto = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(rutaProyecto))

import pandas as pd

import config
from src.analisis import descargarComun, descargarFondo, ejecutarAnalisis, rankingsComunes
from src.catalogoBonos import benchmarkParaFondo
from src.analisisBonos import proyectarInversion
from src.motorPrescriptivo import construirAsignacion, construirContexto, explicacionDeRespaldo, generarExplicacionLLM

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Raíz del proyecto:", rutaProyecto)

## 1. Comprensión del negocio

Un inversor necesita responder: *¿cómo se comportó el fondo?* (descriptivo), *¿qué puede pasar?* (predictivo) y *¿qué hago, según mi perfil?* (prescriptivo, incluidos los bonos que le convienen y cuánto podría ganar).

**Flujo:** perfil → fondo del grupo de ese perfil (por defecto, el mejor del ranking) → análisis completo de ese fondo → fondo completo o bonos específicos de su cartera.

**Criterios de éxito (revisados).**

| Tipo | Criterio | Meta |
|---|---|---|
| Datos | Histórico disponible | ≥ 500 días hábiles |
| Modelo | Habilidad frente a la mejor línea base (MAE en prueba) | > 2% para que el pronóstico se use en la señal |
| Modelo | Cobertura real de la banda del 90% en prueba | entre 80% y 98% |
| Clasificador | F1 macro del RandomForestClassifier | mayor que la persistencia |
| Señal | Backtest semanal | produce REDUCIR en caídas reales |

El MAPE sobre el nivel del NAV ya no es criterio: con retornos diarios de ~1%, cualquier predicción cercana a cero lo cumple.

In [ ]:
PERFIL = "Moderado"
PERIODO = "10y"                             # más historia para el backtest y el clasificador
DIAS = config.DIAS_PRONOSTICO

# Datos comunes: los fondos de bonos de los tres perfiles, la curva del Tesoro y la tasa libre de riesgo
comun = descargarComun(usarCache=True)
rankings = rankingsComunes(comun)

# Igual que la app: el fondo analizado sale del grupo de fondos del perfil (el mejor del ranking),
# salvo que FONDO_POR_DEFECTO en el .env indique otro ticker.
TICKER = config.FONDO_POR_DEFECTO or rankings[PERFIL][0].iloc[0]["ticker"]
BENCHMARK = benchmarkParaFondo(TICKER, PERFIL, config.BENCHMARK_POR_DEFECTO)  # BIL / AGG, o ^SP500TR fuera del catálogo
print(TICKER, BENCHMARK, PERIODO, DIAS, PERFIL)
rankings[PERFIL][0][["puesto", "ticker", "segmento", "veredicto", "puntaje"]]


## 2. Comprensión de los datos

In [ ]:
base = descargarFondo(TICKER, BENCHMARK, PERIODO, config.TICKER_NOTICIAS_POR_DEFECTO, usarCache=True, comun=comun)
historico = base["historico"]
print(f"Registros: {len(historico)} · {historico.index.min():%d/%m/%Y} → {historico.index.max():%d/%m/%Y}")
print(f"Tasa libre de riesgo: {base['tasaLibre']:.2%} ({base['fuenteTasa']})")
print(f"Titulares de {base['tickerNoticias']}: {len(base['noticias'])} · tono de prensa ({base['temaTono']}): {len(base['tono'])} días")
print(f"Bonos descargados: {sum(not d['historico'].empty for d in base['bonos'].values())}")
historico.tail()

## 3. Preparación y análisis completo

Una sola llamada ejecuta preparación, modelado, evaluación, clasificador de riesgo, señal, ranking de bonos y backtest.

In [ ]:
r = ejecutarAnalisis(base, DIAS)
print("Éxito:", r["exito"], r["mensaje"])
for aviso in r["avisos"]:
    print("Aviso:", aviso)
print("Variables del modelo:", r["modelo"].get("columnas"))

## 4. Modelado y 5. Evaluación

### 4.1 Regresión: modelos contra líneas base (Random Forest incluido)

In [ ]:
m = r["modelo"]
print(f"Elegido por validación: {m['mejorNombre']} · predictor usado: {m['nombrePredictor']}")
print(f"Habilidad en prueba: {m['habilidad']:+.1%} · cobertura de la banda 90%: {m['cobertura90']:.0f}%")
m["tabla"]

In [ ]:
r["resumenPronostico"]

### 4.2 Clasificación: riesgo de los próximos 3 meses (RandomForestClassifier)

In [ ]:
c = r["clasificadorRiesgo"]
print(f"Accuracy {c['accuracy']:.2f} · F1 {c['f1Macro']:.2f} | persistencia: {c['accuracyPersistencia']:.2f} · {c['f1Persistencia']:.2f}")
print("Predicción actual:", c["categoriaActual"], c["probabilidades"])
c["matrizConfusion"]

### 5.1 Verificación de los criterios de éxito

In [ ]:
criterios = [
    ("Histórico >= 500 días", len(r["historico"]) >= 500),
    ("Habilidad > 2% frente a la línea base", r["modelo"].get("tieneHabilidad", False)),
    ("Cobertura de la banda entre 80% y 98%", 80 <= r["modelo"].get("cobertura90", 0) <= 98),
    ("Clasificador supera a la persistencia", c["superaPersistencia"]),
    ("El backtest produce REDUCIR", r["resumenBacktest"].get("vecesReducir", 0) > 0),
]
pd.DataFrame([(n, "CUMPLE" if ok else "NO CUMPLE") for n, ok in criterios], columns=["Criterio", "Resultado"])

## 6. Componente prescriptivo

### 6.1 Señal del fondo y sus factores

In [ ]:
s = r["senal"]
print(f"{s['accion']} · puntaje {s['puntaje']:+.1f} · confianza {s['confianza']}")
pd.DataFrame(s["factores"])[["factor", "valor", "puntos", "activo", "nota"]]

### 6.2 Ranking de bonos del perfil y asignación

In [ ]:
tabla, resumen = r["rankingsPorPerfil"][PERFIL]
print(resumen["recomendacion"])
tabla[["puesto", "ticker", "segmento", "veredicto", "puntaje", "rendimientoEsperado", "prima", "volatilidad", "caidaMaxima"]]

In [ ]:
pd.DataFrame(construirAsignacion(PERFIL, s, tabla, claseSenal=r["claseFondo"])["filas"])


### 6.3 Proyección de la ganancia en el mejor bono del perfil

In [ ]:
candidatos = tabla.loc[tabla["rendimientoEsperado"].notna()]
if candidatos.empty:
    print("Sin datos de bonos para proyectar.")
else:
    mejor = candidatos.iloc[0]
    proyeccion = proyectarInversion(r["universoBonos"][mejor["ticker"]]["historico"], mejor["rendimientoEsperado"],
                                    montoInicial=10000, aporteMensual=200, meses=36, tasaLiquidez=r["tasaLibre"])
    print(mejor["ticker"], proyeccion["metodo"])
    display(pd.Series(proyeccion["resumen"]))

### 6.3 bis Bonos individuales dentro del fondo

La cartera bono por bono sale del formulario N-PORT de la SEC. Cada bono se evalúa para el perfil: TIR, duración, diferencial sobre el Tesoro, calidad implícita, rendimiento esperado y veredicto. Requiere `SEC_USER_AGENT` en el `.env`.

In [ ]:
from src.bonosIndividuales import evaluarBonos, evaluarSeleccion, planDeCompra, proyectarBonos, resumirEmisores, cambiosHistoricosTasas
from src.datosSec import descargarCartera, describirMeta

FONDO = tabla.iloc[0]["ticker"]
cartera, meta = descargarCartera(FONDO)
print(describirMeta(meta))
evaluados = evaluarBonos(cartera, PERFIL, r["curva"], r["tasaLibre"], meta.get("fechaReporte"))
display(resumirEmisores(evaluados).head(10))
evaluados[["puesto", "emisor", "cupon", "vencimiento", "precio", "tir", "duracion", "diferencial", "calidad", "rendimientoEsperado", "veredicto", "motivo"]].head(15)


In [ ]:
seleccion = evaluados.loc[evaluados["veredicto"] == "Recomendable"].drop_duplicates("emisor").head(3)
print(evaluarSeleccion(seleccion, PERFIL)["mensaje"])
plan, sobrante = planDeCompra(seleccion, 5000)
plazos, cambios = cambiosHistoricosTasas(r["curvaSeries"], 36)
print(proyectarBonos(plan, sobrante, 36, r["tasaLibre"], plazos, cambios)["resumen"])
plan[["bono", "cusip", "precio", "nominalComprado", "costo", "tir"]]


### 6.4 Backtest de la señal

In [ ]:
rb = r["resumenBacktest"]
print(f"{rb.get('semanas', 0)} semanas · REDUCIR {rb.get('vecesReducir', 0)} veces")
rb.get("porAccion")

### 6.5 Explicación con IA

Requiere `ollama serve` y `ollama pull qwen3:8b` (o `GEMINI_API_KEY`). Las cifras del texto se verifican contra el JSON.

In [ ]:
from IPython.display import Markdown

texto, proveedor, sospechosas = generarExplicacionLLM(construirContexto(r, PERFIL), PERFIL)
if not texto:
    texto = explicacionDeRespaldo(r, PERFIL)
if sospechosas:
    print("Cifras no verificadas:", sospechosas)
Markdown(texto)

In [ ]:
from datetime import datetime

carpeta = rutaProyecto / "data" / "salidas"
carpeta.mkdir(parents=True, exist_ok=True)
marca = datetime.now().strftime("%Y%m%d_%H%M")
(carpeta / f"informe_{TICKER}_{PERFIL}_{marca}.md").write_text(texto, encoding="utf-8")
r["modelo"]["tabla"].to_csv(carpeta / f"modelos_{TICKER}_{marca}.csv", index=False)
r["pronostico"].to_csv(carpeta / f"pronostico_{TICKER}_{marca}.csv")
tabla.drop(columns="componentes").to_csv(carpeta / f"bonos_{PERFIL}_{marca}.csv", index=False)
print("Archivos en", carpeta)

### 6.6 Despliegue

La aplicación repite este flujo al presionar **Actualizar datos**. Levántala en una terminal aparte:

```bash
streamlit run app.py
```

## 7. Monitoreo de calidad y disponibilidad

In [ ]:
for nivel, mensaje in r["calidad"]:
    print(f"[{nivel.upper():6}] {mensaje}")

## Conclusiones

1. **Descriptivo.** Métricas comparadas contra un benchmark de retorno total, para no confundir dividendos con desempeño.
2. **Predictivo.** El pronóstico se evalúa contra líneas base; si ningún modelo las supera, se dice y la señal no lo usa. El Random Forest se conserva en la regresión y en el clasificador de riesgo, que ahora predice el riesgo *futuro*.
3. **Prescriptivo.** Señal auditable por factores, bonos rankeados por perfil (con la opción explícita de *no invertir*), asignación segmentada y proyección de ganancia por escenarios. El LLM explica; no decide.

> Proyecto con fines académicos. No constituye asesoría financiera.